In [ ]:
%cd Semantic

[Errno 2] No such file or directory: 'Semantic'
/content


In [ ]:
!ls

sample_data


**Pulizia dataset training e test**

In [ ]:
import pandas as pd
import json

# --- Rewriting dataset ---
with open("dataset generato/dataset_001/training.json", "r") as f:
    data_rw = [json.loads(line) for line in f]

df_rw = pd.DataFrame(data_rw)

# Rimuovo duplicati e righe con valori nulli
df_rw.dropna(subset=["not_inclusive", "inclusive"], inplace=True)
df_rw.drop_duplicates(subset=["not_inclusive", "inclusive"], inplace=True)

# Salvo dataset pulito
df_rw.to_json("dataset generato/dataset_001/training_clean.json", orient="records", lines=True)


**Pulizia dataset Italiano**

In [ ]:
import pandas as pd
import os

# Percorso corretto usando os.path.join
folder = "dataset controllato"
file_path = os.path.join(folder, "dataset_chatgpt_001.xlsx")

# Caricamento
df = pd.read_excel(file_path)

# Sovrascrivi 'inclusiva' con 'inclusiva_2' SOLO se presente
df["inclusiva"] = df["inclusiva_2"].combine_first(df["inclusiva"])

# Filtra righe dove validazione è "YES"
df = df[df["validazione"].str.upper() == "YES"]

# Rimuovi colonne non richieste
df = df.drop(columns=["inclusiva_2", "note"], errors="ignore")

# Controlla e rimuove righe duplicate (su tutte le colonne)
num_righe_prima = len(df)
df = df.drop_duplicates()
num_righe_dopo = len(df)
duplicati_rimossi = num_righe_prima - num_righe_dopo

print(f"🔎 Righe duplicate rimosse: {duplicati_rimossi}")

# Salvataggio
df.to_json(os.path.join(folder, "dataset_pulito.json"), orient="records", lines=True, force_ascii=False)
df.to_csv(os.path.join(folder, "dataset_pulito.csv"), index=False)

print("Dataset pulito salvato in 'dataset controllato/' come JSON e CSV.")


🔎 Righe duplicate rimosse: 129
✅ Dataset pulito salvato in 'dataset controllato/' come JSON e CSV.


**Augumentation dataset italiano**
API KEY: 

In [ ]:
%pip install --upgrade google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: google-generativeai
    Found existing installation: google-generativeai 0.8.5
    Uninstalling google-generativeai-0.8.5:
      Successfully uninstalled google-generativeai-0.8.5


In [ ]:
import google.generativeai as genai

GOOGLE_API_KEY = ""
genai.configure(api_key=GOOGLE_API_KEY)

print("Elenco modelli disponibili per te:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Elenco modelli disponibili per te:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image-preview
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-202

In [ ]:
!python "dataset generato/augment_dataset_secret_en.py" "dataset controllato/dataset_en.csv" "dataset generato/output" "generatoGemini_en"

[INFO] Pool modelli: 7

[INFO] Categoria: generic (seed: 3641)
   > Batch 1... [OK] +112
   > Batch 2... [ERROR] JSON non valido
   > Batch 3... [OK] +111
   > Batch 4... [OK] +105
   > Batch 5... [OK] +107
   > Batch 6... [OK] +102
   > Batch 7... [ERROR] JSON non valido
   > Batch 8... [ERROR] JSON non valido
   > Batch 9...   [QUOTA] Modello models/gemini-2.5-flash esaurito.

   [SWITCH] Cambio modello -> models/gemini-2.0-flash
   [QUOTA] Modello models/gemini-2.0-flash esaurito.

   [SWITCH] Cambio modello -> models/gemini-2.0-flash-lite
   [QUOTA] Modello models/gemini-2.0-flash-lite esaurito.

   [SWITCH] Cambio modello -> models/gemini-exp-1206
   [QUOTA] Modello models/gemini-exp-1206 esaurito.

   [SWITCH] Cambio modello -> models/gemini-flash-latest
   [QUOTA] Modello models/gemini-flash-latest esaurito.

   [SWITCH] Cambio modello -> models/gemini-2.5-flash-lite
 [OK] +96
   > Batch 10... [OK] +106
   > Batch 11... [OK] +131
   > Batch 12... [OK] +90
   > Batch 13... [ERROR

In [ ]:
!python "/content/Semantic/dataset generato/json2csv.py"

[INFO] Cerco il file: /content/Semantic/dataset generato/output/out_generatoGemini_en.jsonl
[INFO] Lettura file in corso...

[SUCCESSO] Conversione completata.
File salvato in: /content/Semantic/dataset_finale_completo.csv
Totale frasi recuperate: 1690


**Pulizia csv**

In [ ]:
import pandas as pd
import os

# --- CONFIGURAZIONE PERCORSI ---
input_file = '/content/Semantic/dataset_en.csv'
output_folder = '/content/Semantic'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

print(f"[INFO] Caricamento file: {input_file}")

try:
    # Caricamento (gestisce sia virgola che punto e virgola in ingresso)
    try:
        df = pd.read_csv(input_file, encoding='utf-8-sig', sep=None, engine='python')
    except:
        df = pd.read_csv(input_file, encoding='utf-8', sep=None, engine='python')

    print(f"   Righe iniziali: {len(df)}")

    # 1. NORMALIZZAZIONE NOMI COLONNE
    df.columns = df.columns.str.strip()

    # 2. SELEZIONE COLONNE
    cols_to_keep = ['non_inclusiva', 'inclusiva']
    existing_cols = [c for c in cols_to_keep if c in df.columns]
    df = df[existing_cols]

    # 3. PULIZIA
    df = df.dropna(how='all')
    if 'non_inclusiva' in df.columns:
        df = df.dropna(subset=['non_inclusiva'])

    # 4. RIMOZIONE DUPLICATI
    num_righe_prima = len(df)
    df = df.drop_duplicates(subset=['non_inclusiva'])
    num_righe_dopo = len(df)

    print(f"Righe duplicate rimosse: {num_righe_prima - num_righe_dopo}")

    # 5. SALVATAGGIO (MODIFICATO PER EXCEL ITALIANO)
    json_out = os.path.join(output_folder, "dataset_generato3p_en.json")
    csv_out = os.path.join(output_folder, "dataset_generato3p_en.csv")

    df.to_json(json_out, orient="records", lines=True, force_ascii=False)

    # --- MODIFICA QUI: sep=';' ---
    # Questo dice a Pandas di usare il punto e virgola, che Excel italiano adora.
    df.to_csv(csv_out, index=False, encoding='utf-8-sig', sep=';')

    print(f"Dataset pulito salvato con {len(df)} righe finali.")
    print(f" - CSV (Excel Ready): {csv_out}")
    print(f" - JSON: {json_out}")

except Exception as e:
    print(f"[ERRORE] {e}")

[INFO] Caricamento file: /content/Semantic/dataset_en.csv
   Righe iniziali: 5331
🔎 Righe duplicate rimosse: 351
✅ Dataset pulito salvato con 4980 righe finali.
 - CSV (Excel Ready): /content/Semantic/dataset_generato3p_en.csv
 - JSON: /content/Semantic/dataset_generato3p_en.json


**Lavoro sul dataset inglese**

In [ ]:
import os

BASE = "/content/Semantic"
CSV_PATH = os.path.join(BASE, "dataset_en.csv")
JSON_PATH = os.path.join(BASE, "dataset_en.json")

print("Exists CSV:", os.path.exists(CSV_PATH), CSV_PATH)
print("Exists JSON:", os.path.exists(JSON_PATH), JSON_PATH)

print("\nFolder content:")
print(os.listdir(BASE))


Exists CSV: True /content/Semantic/dataset_en.csv
Exists JSON: True /content/Semantic/dataset_en.json

Folder content:
['.git', '.gitignore', 'training_code', 'dataset controllato', 'struttura.txt', 'dataset_en.json', 'dataset_en.csv', 'documentazione', 'dataset generato', 'webapp']


In [ ]:
import pandas as pd
import json

def load_any_dataset(csv_path, json_path):
    if os.path.exists(csv_path):
        # prova separatore ; (tipico) poi ,
        try:
            df = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
        except Exception:
            df = pd.read_csv(csv_path, sep=",", encoding="utf-8-sig")
        src = "csv"
        return df, src

    if os.path.exists(json_path):
        # supporta JSON array oppure JSONL
        with open(json_path, "r", encoding="utf-8") as f:
            first = f.read(2048).lstrip()
        if first.startswith("["):
            df = pd.read_json(json_path)
        else:
            rows = []
            with open(json_path, "r", encoding="utf-8") as f:
                for line in f:
                    line=line.strip()
                    if line:
                        rows.append(json.loads(line))
            df = pd.DataFrame(rows)
        src = "json"
        return df, src

    raise FileNotFoundError("Non trovo né dataset_en.csv né dataset_en.json in /content/Semantics")

df, src = load_any_dataset(CSV_PATH, JSON_PATH)
print("Loaded from:", src)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)


Loaded from: csv
Shape: (4980, 2)
Columns: ['non_inclusiva', 'inclusiva']


,non_inclusiva,inclusiva
0,Blonde women are usually not very smart.,Hair color does not determine a person's intel...
1,He's too sensitive for a man.,"Expressing emotions is a human trait, regardle..."
2,Girls only care about shopping and makeup.,"Interests vary greatly among individuals, rega..."


In [ ]:
import numpy as np

# possibili nomi colonne
CAND_NOT = ["non_inclusiva", "not_inclusive", "non_inclusive_sentence", "source", "input", "original", "biased"]
CAND_INC = ["inclusiva", "inclusive", "inclusive_sentence", "target", "output", "rewritten", "neutral"]

def pick_col(cols, candidates):
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

col_not = pick_col(df.columns, CAND_NOT)
col_inc = pick_col(df.columns, CAND_INC)

print("Detected not_inclusive column:", col_not)
print("Detected inclusive column:", col_inc)

assert col_not is not None and col_inc is not None, "Non riesco ad auto-detectare le colonne. Incolla qui Columns e le prime righe."

pairs = df[[col_not, col_inc]].copy()
pairs.columns = ["not_inclusive", "inclusive"]

# pulizia base
pairs["not_inclusive"] = pairs["not_inclusive"].astype(str).str.strip()
pairs["inclusive"] = pairs["inclusive"].astype(str).str.strip()

pairs = pairs.replace({"nan": np.nan, "None": np.nan, "": np.nan})
pairs = pairs.dropna(subset=["not_inclusive", "inclusive"]).drop_duplicates()

print("Pairs after cleaning:", pairs.shape)
pairs.head(3)


Detected not_inclusive column: non_inclusiva
Detected inclusive column: inclusiva
Pairs after cleaning: (4980, 2)


,not_inclusive,inclusive
0,Blonde women are usually not very smart.,Hair color does not determine a person's intel...
1,He's too sensitive for a man.,"Expressing emotions is a human trait, regardle..."
2,Girls only care about shopping and makeup.,"Interests vary greatly among individuals, rega..."


In [ ]:
from sklearn.model_selection import train_test_split
import json, os

OUT_DIR = os.path.join(BASE, "dataset_002")
os.makedirs(OUT_DIR, exist_ok=True)

train_pairs, test_pairs = train_test_split(pairs, test_size=0.20, random_state=42, shuffle=True)

print("Train pairs:", train_pairs.shape)
print("Test pairs:", test_pairs.shape)

# --- REWRITE files (train.json / test.json) ---
def save_jsonl(df_, path, mode):
    with open(path, "w", encoding="utf-8") as f:
        for _, r in df_.iterrows():
            if mode == "rewrite":
                obj = {"not_inclusive": r["not_inclusive"], "inclusive": r["inclusive"]}
            else:
                raise ValueError("unknown mode")
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")

save_jsonl(train_pairs, os.path.join(OUT_DIR, "train.json"), "rewrite")
save_jsonl(test_pairs,  os.path.join(OUT_DIR, "test.json"),  "rewrite")

# --- CLASSIFICATION files (train_classification.json / test_classification.json) ---
# dal rewriting creiamo 2 esempi per coppia: uno inclusive e uno not_inclusive
def make_classification(df_):
    rows = []
    for _, r in df_.iterrows():
        rows.append({"text": r["inclusive"], "label": "inclusive", "type": "en", "sub_type": ""})
        rows.append({"text": r["not_inclusive"], "label": "not_inclusive", "type": "en", "sub_type": ""})
    return pd.DataFrame(rows)

train_clf = make_classification(train_pairs).sample(frac=1, random_state=42).reset_index(drop=True)
test_clf  = make_classification(test_pairs).sample(frac=1, random_state=42).reset_index(drop=True)

def save_jsonl_generic(df_, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, r in df_.iterrows():
            f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

save_jsonl_generic(train_clf, os.path.join(OUT_DIR, "train_classification.json"))
save_jsonl_generic(test_clf,  os.path.join(OUT_DIR, "test_classification.json"))

print("\nSaved files in:", OUT_DIR)
print(os.listdir(OUT_DIR))

print("\nLabel distribution (train):")
print(train_clf["label"].value_counts())

print("\nLabel distribution (test):")
print(test_clf["label"].value_counts())


Train pairs: (3984, 2)
Test pairs: (996, 2)

Saved files in: /content/Semantic/dataset_002
['train.json', 'train_classification.json', 'test_classification.json', 'test.json']

Label distribution (train):
label
inclusive        3984
not_inclusive    3984
Name: count, dtype: int64

Label distribution (test):
label
not_inclusive    996
inclusive        996
Name: count, dtype: int64
